# Kernel Evaluation Notebook

In this notebook we provide a step-by-step guide for evaluating the performance of a single custom generated kernel against a single KernelBench problem.

**Supported devices:** Intel XPU or Nvidia GPU

**Prerequisites:** 
1. Custom kernel: create your own kernel for a single KernelBench problem (verifiede on triton kernels only)
2. The provided custom kernel must have a **ModelNew wrapper** including a forward function that runs the kernel
3. Set the following arguments inside this notebook before running it: problem_id, problem_level, custom_kernel_path, use_xpu (default: True)

**Output:** Speedup of provided custom kernel vs. pytorch eager/compile

## Installations

In [ ]:
# ! git clone https://github.com/ScalingIntelligence/KernelBench.git
# % cd KernelBench
# ! pip install -e .

# installations for XPU:
# ! pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/xpu

# installations for running jupyter notebook:
# !pip install -U jupyterlab jupyter ipywidgets

In [ ]:
from datasets import load_dataset
import os
import random
import tempfile
import traceback
import numpy as np
import torch
import torch.nn as nn
from torch.profiler import profile, ProfilerActivity
import torch.fx as fx
from torch.fx.passes.shape_prop import ShapeProp
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from src.kernelbench.utils import (
    set_gpu_arch,
)
from kernelbench.eval import (
    load_original_model_and_inputs,
    load_custom_model_with_tempfile,
    set_seed,
    _process_input_tensor,
    get_tolerance_for_precision
)

## Configurations

In [ ]:
problem_id = # set problem id here
problem_level = # set KernelBench problem level here
custom_kernel_path = # set custom kernel path here

In [ ]:
# precision = torch.float32
precision = torch.bfloat16

In [ ]:
dataset_name = "ScalingIntelligence/KernelBench"
seed_num = 42
num_warmup = 100
num_time_trials = 200
num_correctness_trials = 5
backend = "triton"

torch.set_printoptions(
        precision=4,  # Decimal places
        threshold=10,  # Total number of elements before truncating
        edgeitems=3,  # Number of elements at beginning and end of dimensions
        linewidth=80,  # Maximum width before wrapping
    )

### Device configuration

In [ ]:
use_xpu = True

if use_xpu:
    device = torch.device("xpu")
else:
    gpu_arch = ["Ada"]
    set_gpu_arch(gpu_arch)
    
    assert torch.cuda.is_available(), "CUDA is not available, cannot run Eval"
    device = torch.cuda.current_device()
    
    # set CUDA device
    torch.cuda.set_device(device)
    
    # need to set env var for triton/cute code to guarantee no wrong device shenanigans
    if isinstance(device, int):
        device_num = device
    elif isinstance(device, torch.device):
        assert device.type == "cuda", "CUDA is not availible on device, cannot run Eval"
        device_num = device.index
    else:
        raise ValueError(f"device must be an int or torch.device, got {type(device)}")
    os.environ["CUDA_VISIBLE_DEVICES"] = str(device_num)

## Load KernelBench problem to evaluate

In [ ]:
dataset = load_dataset(dataset_name)
curr_level_dataset = dataset[f"level_{problem_level}"]
curr_problem_row = curr_level_dataset.filter(lambda x: x["problem_id"] == problem_id)
ref_arch_src = curr_problem_row["code"][0]
problem_name = curr_problem_row["name"][0]

## Load/verify original model and inputs

In [ ]:
context = {}

Model, get_init_inputs, get_inputs = load_original_model_and_inputs(ref_arch_src, context)
set_seed(seed_num)  # set seed for reproducible input
init_inputs = get_init_inputs()
# Convert inputs to appropriate dtypes for GPU computation
init_inputs = [_process_input_tensor(x, device, backend, precision) for x in init_inputs]

with torch.no_grad():
    set_seed(seed_num)  # set seed for reproducible weights
    original_model = Model(*init_inputs)
    assert hasattr(original_model, "forward")

## Load/verify the custom kernel

In [25]:
if not use_xpu:
    os.environ["TORCH_USE_CUDA_DSA"] = "1"  # compile with device side assertion

temp_file = None

with open(custom_kernel_path, "r", encoding="utf-8") as f:
    custom_kernel = f.read()
    
ModelNew, temp_file = load_custom_model_with_tempfile(
                custom_kernel, entry_point="ModelNew"
            )
with torch.no_grad():
    set_seed(seed_num)  # set seed for reproducible weights
    custom_model = ModelNew(*init_inputs)
    assert hasattr(custom_model, "forward")

## Prepare inputs for speedup measurement:

In [ ]:
set_seed(seed_num)
inputs = get_inputs()
# Convert inputs for performance measurement

inputs = [_process_input_tensor(x, device, backend, precision) for x in inputs]


### Define helper functions

In [ ]:
def synchronize(device):
    if device.type == "xpu":
        torch.xpu.synchronize(device=device)
    else:
        torch.cuda.synchronize(device=device)


## Measure runtime of original (pytorch) model

In [ ]:
model = original_model.to(device=device, dtype=precision)
synchronize(device)
# torch.cuda.synchronize(device=device)

# Warm ups
for _ in range(num_warmup):
    model(*inputs)
    synchronize(device)

original_model_times = []

# Actual trials
for trial in range(num_time_trials):
    # create event marker default is not interprocess
    if device.type == 'xpu':
        start_event = torch.xpu.Event(enable_timing=True)
        end_event = torch.xpu.Event(enable_timing=True)
    else:
        start_event = torch.cuda.Event(enable_timing=True)
        end_event = torch.cuda.Event(enable_timing=True)

    
    start_event.record()
    model(*inputs)
    end_event.record()

    # Synchronize to ensure the events have completed
    synchronize(device)

    # Calculate the elapsed time in milliseconds
    elapsed_time_ms = start_event.elapsed_time(end_event)
    original_model_times.append(elapsed_time_ms)

orig_runtime = np.mean(original_model_times)
print(f"\nOriginal model average time: {orig_runtime:.3g} ms")


## Measure runtime of compiled model:

In [ ]:
compiled_model = torch.compile(model).to(device=device, dtype=precision)


synchronize(device)

# Warm ups
for _ in range(num_warmup):
    compiled_model(*inputs)
    synchronize(device)

compiled_model_times = []

# Actual trials
for trial in range(num_time_trials):
    # create event marker default is not interprocess
    if device.type == 'xpu':
        start_event = torch.xpu.Event(enable_timing=True)
        end_event = torch.xpu.Event(enable_timing=True)
    else:
        start_event = torch.cuda.Event(enable_timing=True)
        end_event = torch.cuda.Event(enable_timing=True)

    start_event.record()
    compiled_model(*inputs)
    end_event.record()

    # Synchronize to ensure the events have completed
    synchronize(device)

    # Calculate the elapsed time in milliseconds
    elapsed_time_ms = start_event.elapsed_time(end_event)
    compiled_model_times.append(elapsed_time_ms)

compiled_runtime = np.mean(compiled_model_times)
print(f"\nCompiled model average time: {compiled_runtime:.3g} ms")




## Check correctness of custom kernel
Before evaluating the performance of the custom kernel we must ensure that it is correct:

In [ ]:
if not use_xpu:
    os.environ["TORCH_USE_CUDA_DSA"] = "1"  # compile with device side assertion

pass_count = 0
# Generate num_correct_trials seeds deterministically from the initial seed
torch.manual_seed(seed_num)
correctness_trial_seeds = [
    torch.randint(0, 2**32 - 1, (1,)).item() for _ in range(num_correctness_trials)
]

# in torchbench, they use both precisions for atol and rtol
# kernelbench v0 and v0.1 uses fp32, atol = rtol = 1e-02
# now we will return the tolerance from get_tolerance_for_precision
tolerance = get_tolerance_for_precision(precision)
print(f"tolerance: {tolerance}")

with torch.no_grad():
    for trial in range(num_correctness_trials):
        trial_seed = correctness_trial_seeds[trial]
        set_seed(trial_seed)
        
        temp_file = None
        
        with open(custom_kernel_path, "r", encoding="utf-8") as f:
            custom_kernel = f.read()
            
        ModelNew, temp_file = load_custom_model_with_tempfile(
                        custom_kernel, entry_point="ModelNew"
                    )
        with torch.no_grad():
            set_seed(trial_seed)
            custom_model = ModelNew(*init_inputs)
            assert hasattr(custom_model, "forward")
        
        context = {}
        
        Model, get_init_inputs, get_inputs = load_original_model_and_inputs(ref_arch_src, context)
        set_seed(seed_num)  # set seed for reproducible input
        init_inputs = get_init_inputs()
        # Convert inputs to appropriate dtypes for GPU computation
        init_inputs = [_process_input_tensor(x, device, backend, precision) for x in init_inputs]
        
        with torch.no_grad():
            set_seed(trial_seed)
            original_model = Model(*init_inputs)
            assert hasattr(original_model, "forward")
        
        inputs = get_inputs()
        
        # Convert inputs to appropriate dtypes for GPU computation
        inputs = [_process_input_tensor(x, device, backend, precision) for x in inputs]

        set_seed(trial_seed)
        model = original_model.to(device=device, dtype=precision)
        set_seed(trial_seed)
        model_new = custom_model.to(device=device, dtype=precision)

        output_orig = model(*inputs)
        synchronize(device)
        output_new = model_new(*inputs)
        synchronize(device)
        # ensure all GPU operations are completed before checking results
        
        if trial==0: # print output dtype
            print(f"output dtype: {output_orig.dtype}")
            print(f"output_new dtype: {output_new.dtype}")
            
        if output_orig.shape != output_new.shape:
            print(f"Output shape mismatch: Expected {output.shape}, got {output_new.shape}")
            
        # break        
        # cast to bf16
        output_orig = output_orig.bfloat16()
        output_new = output_new.bfloat16()
        
        # check output value difference
        
        max_diff = torch.max(torch.abs(output_orig - output_new)).item()
        avg_diff = torch.mean(torch.abs(output_orig - output_new)).item()
        
        if not torch.allclose(output_orig, output_new, atol=tolerance, rtol=tolerance):  # fail
            print(f"Failed trial {trial}: Output mismatch. max_difference: {max_diff:.6f}, avg_difference: {avg_diff:.6f}")
        else:  # pass
            pass_count += 1
            print(f"[PASS] trial {trial}: ModelNew matches Model! max_difference: {max_diff:.6f}, avg_difference: {avg_diff:.6f}")

if pass_count == num_correctness_trials:
    print(f"\nKernel passed correctness check !")
else:
    print(f"\nKernel didn't passed correctness check !")


## Measure runtime of custom kernel
In case the custom kernel is correct, we will now measure its runtime:

In [ ]:
new_model = custom_model.to(device=device, dtype=precision)
synchronize(device)

# Warm ups
for _ in range(num_warmup):
    new_model(*inputs)
    synchronize(device)

new_model_times = []

# Actual trials
for trial in range(num_time_trials):
    # create event marker default is not interprocess
    if device.type == 'xpu':
        start_event = torch.xpu.Event(enable_timing=True)
        end_event = torch.xpu.Event(enable_timing=True)
    else:
        start_event = torch.cuda.Event(enable_timing=True)
        end_event = torch.cuda.Event(enable_timing=True)

    start_event.record()
    new_model(*inputs)
    end_event.record()

    # Synchronize to ensure the events have completed
    synchronize(device)

    # Calculate the elapsed time in milliseconds
    elapsed_time_ms = start_event.elapsed_time(end_event)
    new_model_times.append(elapsed_time_ms)

custom_runtime = np.mean(new_model_times)
print(f"\nCustom model average time: {custom_runtime:.3g} ms")


## Print speedup results
We can finally report the total speedup:

In [ ]:
print(f"speedup custom model vs. pytorch model = {(orig_runtime / custom_runtime):.3g}x")
print(f"speedup torch.compiled model vs. pytorch model = {(orig_runtime / compiled_runtime):.3g}x")
print(f"speedup custom model vs. torch.compiled model = {(compiled_runtime / custom_runtime):.3g}x")

### Support

For any issues/comments please write to shira.guskin@intel.com